# 🚁 Tello Explorer — Notebook Sim + Vol Réel

Exploration autonome de bâtiments délabrés avec **cartographie thermique**, pour drone **DJI Tello EDU**.

Ce notebook couvre **deux modes**, exécutables indépendamment :

- **Partie A — Simulation** (sections 1→6) : aucun matériel requis, frames synthétiques.
- **Partie B — Vol réel** (sections 7→10) : drone connecté en Wi-Fi sur `192.168.10.1`.

> Réglages alignés sur le notebook de terrain : connexion par `host="192.168.10.1"`,
> séquence `streamon()` → `time.sleep(2)` → `get_frame_read()`, frames natives **960×720** (shape `(720, 960, 3)`),
> conversion **BGR→RGB** pour l'affichage uniquement, `djitellopy >= 2.5.0`.

---

## 0. Installation des dépendances

In [ ]:
# Décommenter pour installer (ou: pip install -r requirements.txt)
# !pip install -q "djitellopy>=2.5.0" numpy opencv-python matplotlib

### Imports & configuration du logging

In [ ]:
import sys, os, time, logging
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Modules du projet (doivent être dans le même dossier que ce notebook)
from tello_controller import TelloController, DroneState, DEFAULT_TELLO_HOST
from exploration import ExplorationMission, MissionConfig, MissionStatus
from vision import VideoStream, ObstacleDetector, ThermalDetector
from mapping import DualMap, ExplorationPlanner
from obstacle_avoidance import ObstacleAvoidanceSystem, ThreatLevel

print("✅ Modules chargés")
print(f"📁 Répertoire : {os.getcwd()}")
print(f"🌐 Host drone par défaut : {DEFAULT_TELLO_HOST}")

---
# Partie A — SIMULATION

*Aucun drone requis. Tout tourne en `simulation_mode=True`.*

## 1. Test du contrôleur (simulation)

In [ ]:
controller = TelloController(simulation_mode=True)
controller.connect()
print(f"État initial : {controller.state.value}")
print(f"Position     : {controller.position.to_tuple()}")

controller.takeoff()
print(f"\nAprès décollage : {controller.position.to_tuple()}")

movements = [
    ("Avance 100cm", lambda: controller.move_forward(100)),
    ("Rotation 90°", lambda: controller.rotate_clockwise(90)),
    ("Avance 50cm",  lambda: controller.move_forward(50)),
    ("Monte 30cm",   lambda: controller.move_up(30)),
]
print("\nSéquence de mouvements :")
for name, action in movements:
    action()
    p = controller.position
    print(f"  {name:14s} → ({p.x:.0f}, {p.y:.0f}, {p.z:.0f}) | yaw={p.yaw:.0f}°")

print("\nTélémétrie :", controller.get_telemetry())
controller.land(); controller.disconnect()
print("✅ Test contrôleur terminé")

## 2. Vision & détection (simulation)

In [ ]:
import cv2

video    = VideoStream(simulation_mode=True)
detector = ObstacleDetector()
thermal  = ThermalDetector()

video.start(); time.sleep(1)
frame = video.get_frame()

if frame is not None:
    obstacles = detector.detect(frame)
    thermal_map, hotspots = thermal.detect(frame)

    print(f"Frame          : shape={frame.shape}  (attendu (720, 960, 3))")
    print(f"Obstacles      : {len(obstacles)}")
    print(f"Points chauds  : {len(hotspots)}")
    print(f"Température max : {thermal.get_max_temperature():.1f}°C")

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    # Conversion BGR→RGB pour matplotlib (cf. notebook terrain)
    axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title("Vue caméra (simulée)"); axes[0].axis('off')

    det = detector.draw_detections(frame, obstacles)
    axes[1].imshow(cv2.cvtColor(det, cv2.COLOR_BGR2RGB))
    axes[1].set_title(f"Obstacles ({len(obstacles)})"); axes[1].axis('off')

    if thermal_map is not None:
        axes[2].imshow(cv2.cvtColor(thermal_map, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f"Thermique (max {thermal.get_max_temperature():.0f}°C)"); axes[2].axis('off')
    plt.tight_layout(); plt.show()

video.stop()
print("✅ Test vision terminé")

## 3. Configuration de la mission (simulation)

In [ ]:
config = MissionConfig(
    area_width=300, area_height=300,   # zone 3m × 3m
    exploration_altitude=100,          # 1 m
    step_size=50,                      # 50 cm entre points
    pattern="snake",                   # snake | spiral | room_search
    scan_interval=200,                 # scan 360° tous les 2 m
    safety_margin=80,
    min_battery=15,
    max_duration=120,
    host=DEFAULT_TELLO_HOST,           # 192.168.10.1 (ignoré en simulation)
    enable_mapping=True, enable_thermal=True,
    enable_avoidance=True, enable_scanning=True,
)
print("Zone        :", config.area_width/100, "×", config.area_height/100, "m")
print("Altitude    :", config.exploration_altitude, "cm")
print("Pattern     :", config.pattern)
print("Scan tous   :", config.scan_interval, "cm")

### 3.1 Préparation + callbacks

In [ ]:
mission = ExplorationMission(config, simulation_mode=True)

mission_events, waypoints_reached, thermal_alerts = [], [], []

def on_status(old, new):
    print(f"[{time.strftime('%H:%M:%S')}] {old.value} → {new.value}")
def on_waypoint(wp, progress):
    waypoints_reached.append(wp)
    if len(waypoints_reached) % 5 == 0:
        print(f"  ✓ ({wp[0]:.0f}, {wp[1]:.0f}) — {progress:.1f}%")
def on_thermal(pos, temp, hotspots):
    thermal_alerts.append((pos, temp, time.time()))
    print(f"  🔥 {temp:.0f}°C à ({pos.x:.0f}, {pos.y:.0f})")

mission.on_status_change   = on_status
mission.on_waypoint_reached = on_waypoint
mission.on_thermal_alert   = on_thermal

if mission.prepare_mission():
    print(f"\n✅ Mission prête — {mission.total_waypoints} waypoints")
else:
    print("❌ Échec de la préparation")

### 3.2 Environnement simulé (obstacles + zones thermiques)

In [ ]:
# Obstacles
mission.add_simulated_obstacle(80, 50, 100, obstacle_type="debris")
mission.add_simulated_obstacle(-60, 80, 100, obstacle_type="debris")
mission.add_simulated_obstacle(0, 120, 100, is_mobile=True, velocity=(8, 3, 0), obstacle_type="person")
mission.dual_map.add_obstacle(-30, -50, 0, radius=40, obstacle_type="hole", threat_level=4)

# Zones thermiques
mission.dual_map.add_thermal_zone(100, 80, 50, radius=60, temperature=180, is_active=True)   # feu
mission.dual_map.add_thermal_zone(-80, 60, 30, radius=40, temperature=85,  is_active=False)  # braises
mission.dual_map.add_thermal_zone(50, -70, 20, radius=50, temperature=45,  is_active=False)  # tiède
print("✅ Environnement configuré")

## 4. Exécution de la mission (simulation)

In [ ]:
mission.start_exploration()

start = time.time(); duration = 15
try:
    while mission.status == MissionStatus.IN_PROGRESS and (time.time()-start) < duration:
        time.sleep(1)
        if int(time.time()-start) % 5 == 0:
            p = mission.controller.position
            print(f"  T+{int(time.time()-start)}s : ({p.x:.0f}, {p.y:.0f}, {p.z:.0f}) | {mission.planner.get_progress():.1f}%")
except KeyboardInterrupt:
    print("⚠️ Interruption")

mission.stop_exploration()
print(f"\nWaypoints atteints : {len(waypoints_reached)} | Alertes thermiques : {len(thermal_alerts)}")

## 5. Visualisation des résultats (simulation)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
ext = [-config.area_width/2, config.area_width/2, -config.area_height/2, config.area_height/2]

# Altitude
ax = axes[0,0]
im = ax.imshow(np.nan_to_num(mission.dual_map.altitude_grid, nan=0), cmap='terrain', origin='lower', extent=ext)
plt.colorbar(im, ax=ax, label='Altitude (cm)'); ax.set_title("Carte d'altitude")
for o in mission.dual_map.obstacles:
    ax.plot(o.x, o.y, 'X', color=('red' if o.is_mobile else 'black'), ms=10)

# Thermique
ax = axes[0,1]
im = ax.imshow(mission.dual_map.thermal_grid, cmap='hot', origin='lower', extent=ext, vmin=20, vmax=200)
plt.colorbar(im, ax=ax, label='°C'); ax.set_title("Carte thermique")
for z in mission.dual_map.thermal_zones:
    ax.add_patch(plt.Circle((z.x, z.y), z.radius, fill=False, color='cyan', lw=2))

# Occupation
ax = axes[1,0]
from matplotlib.colors import ListedColormap, BoundaryNorm
cmap = ListedColormap(['lightgray','lightgreen','red']); norm = BoundaryNorm([-1.5,-0.5,0.5,100.5], cmap.N)
ax.imshow(mission.dual_map.occupancy_grid, cmap=cmap, norm=norm, origin='lower', extent=ext)
ax.set_title("Grille d'occupation")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(facecolor='lightgray',label='Inexploré'),
                   Patch(facecolor='lightgreen',label='Libre'),
                   Patch(facecolor='red',label='Occupé')], loc='upper right')

# Trajectoire
ax = axes[1,1]
if waypoints_reached:
    tx = [w[0] for w in waypoints_reached]; ty = [w[1] for w in waypoints_reached]
    ax.plot(tx, ty, 'b-', alpha=.6, lw=1)
    ax.scatter(tx, ty, c=np.linspace(0,1,len(tx)), cmap='viridis', s=20)
    ax.plot(tx[0], ty[0], 'go', ms=12, label='Départ')
    ax.plot(tx[-1], ty[-1], 'r^', ms=12, label='Fin')
for o in mission.dual_map.obstacles:
    ax.add_patch(plt.Circle((o.x, o.y), o.radius, color=('red' if o.is_mobile else 'orange'), alpha=.5))
ax.set_xlim(ext[0]-50, ext[1]+50); ax.set_ylim(ext[2]-50, ext[3]+50)
ax.set_aspect('equal'); ax.grid(alpha=.3); ax.legend(loc='upper right'); ax.set_title("Trajectoire")

for a in axes.flat: a.set_xlabel('X (cm)'); a.set_ylabel('Y (cm)')
plt.tight_layout(); plt.show()

### 5.1 Cartes ASCII + rapport

In [ ]:
print(mission.dual_map.to_ascii_map((mission.controller.position.x, mission.controller.position.y), show_thermal=False))
print()
print(mission.dual_map.to_ascii_map((mission.controller.position.x, mission.controller.position.y), show_thermal=True))

In [ ]:
r = mission.get_mission_report()
print(f"Statut          : {r['status']}")
print(f"Durée           : {r['duration_seconds']:.1f}s")
print(f"Waypoints       : {r['waypoints']['completed']}/{r['waypoints']['total']} ({r['waypoints']['progress']:.1f}%)")
print(f"Couverture      : {r['mapping']['coverage']:.1f}%")
print(f"Obstacles       : {r['mapping']['obstacles_count']} (mobiles: {r['mapping']['mobile_obstacles']})")
print(f"Température max  : {r['thermal']['max_temperature']:.1f}°C | feu: {r['thermal']['fire_detected']}")
print(f"Zones thermiques: {r['thermal']['zones']}")
print(f"Batterie        : {r['drone']['battery']}%")

## 6. Export des données (simulation)

In [ ]:
out = "/tmp/exploration_data"; os.makedirs(out, exist_ok=True)
mission.export_results(f"{out}/mission")
print("Fichiers générés :")
for f in sorted(os.listdir(out)):
    print(f"  📄 {f} ({os.path.getsize(f'{out}/{f}')/1024:.1f} KB)")

---
# Partie B — VOL RÉEL

⚠️ **Sécurité** : zone dégagée, hélices intactes, batterie chargée, garde une main sur l'arrêt d'urgence.

**Connexion** : ton PC doit être associé au Wi-Fi du Tello (`TELLO-XXXXXX`).

> **À propos du host.** En **mode point d'accès** (le Tello crée son propre Wi-Fi), son IP est `192.168.10.1`.
> Le code laisse désormais `host=None` par défaut → djitellopy utilise automatiquement `192.168.10.1`.
> Ne renseigne une IP (ex. `192.168.0.101`) **que** si ton Tello est en *mode station* sur ton routeur.
>
> **Une seule connexion à la fois.** Le Tello n'accepte qu'**un client UDP**. Les sections 7–9 ouvrent
> une instance `drone` manuelle ; la section 10 la **réutilise** (au lieu d'en créer une seconde, ce qui
> provoquait les timeouts `Aborting command 'command'`).

## 7. Connexion & batterie (réel)

### Configuration réseau (source unique de vérité)

Définis **ici** l'adresse du drone, réutilisée par toutes les cellules de la Partie B.

- **Mode station** (Tello connecté à ton routeur, PC sur le même réseau `192.168.0.x`) :
  mets son IP, ex. `"192.168.0.101"`.
- **Mode point d'accès** (PC connecté au Wi-Fi `TELLO-XXXXXX`) : mets `None`
  → djitellopy utilise `192.168.10.1`.

In [ ]:
# Nettoyage défensif avant toute nouvelle connexion
for _name in ("mission_real", "mission"):
    _m = globals().get(_name)
    if _m is not None:
        try: _m.stop_exploration()
        except Exception: pass
_d = globals().get("drone")
if _d is not None:
    try: _d.end()
    except Exception: pass
print("État nettoyé — prêt pour une nouvelle connexion")

In [ ]:
# ⬇️ UNE SEULE adresse à régler pour toute la Partie B
TELLO_HOST = "192.168.0.101"   # mode station. Mets None pour le mode point d'accès (192.168.10.1).

print("Host drone utilisé :", TELLO_HOST if TELLO_HOST else "auto djitellopy (192.168.10.1)")

In [ ]:
from djitellopy import Tello

# Réutilise l'adresse définie dans la cellule de configuration ci-dessus.
# host=None => défaut djitellopy (192.168.10.1) ; sinon IP explicite (mode station).
drone = Tello(host=TELLO_HOST) if TELLO_HOST else Tello()
drone.connect()
print(f"Batterie : {drone.get_battery()}%")
print(f"Hauteur  : {drone.get_height()} cm")
print(f"Temp     : {drone.get_temperature()} °C")
print(f"Host     : {drone.address}")

## 8. Flux vidéo live (réel)

Séquence du notebook terrain : `streamon()` → `time.sleep(2)` *(TRÈS IMPORTANT)* → `get_frame_read()`. La fenêtre OpenCV s'ouvre ; appuie sur **`q`** pour arrêter.

In [ ]:
import cv2

drone.streamon()
time.sleep(2)                      # TRÈS IMPORTANT (stabilisation du flux)
frame_reader = drone.get_frame_read()

try:
    while True:
        frame_bgr = frame_reader.frame
        if frame_bgr is None:
            continue
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)   # BGR→RGB pour affichage
        cv2.imshow("Tello Live Video", frame_rgb)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
finally:
    drone.streamoff()
    cv2.destroyAllWindows()

print("✅ Flux arrêté — shape dernière frame :", None if frame_bgr is None else frame_bgr.shape)

### 8.1 Variante notebook : une frame analysée (sans fenêtre OpenCV)
Pratique si `cv2.imshow` ne s'affiche pas (environnement headless / Jupyter distant).

In [ ]:
import cv2
detector = ObstacleDetector(); thermal = ThermalDetector()

frame_bgr = frame_reader.frame                      # déjà démarré en 8.
obstacles = detector.detect(frame_bgr)
tmap, hotspots = thermal.detect(frame_bgr)

print(f"shape={frame_bgr.shape} | obstacles={len(obstacles)} | hotspots={len(hotspots)} | Tmax={thermal.get_max_temperature():.0f}°C")

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].imshow(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)); ax[0].set_title("Live"); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(detector.draw_detections(frame_bgr, obstacles), cv2.COLOR_BGR2RGB))
ax[1].set_title(f"Détections ({len(obstacles)})"); ax[1].axis('off')
plt.tight_layout(); plt.show()

## 9. Décollage / mouvements / atterrissage (réel)

Test minimal de vol, dans l'esprit de la cellule `takeoff → sleep → land` du notebook de terrain.

In [ ]:
try:
    drone.takeoff()
    time.sleep(2)
    # Décommenter pour un petit carré (zone TRÈS dégagée requise) :
    # for _ in range(4):
    #     drone.move_forward(50); time.sleep(1)
    #     drone.rotate_clockwise(90); time.sleep(1)
    drone.land()
except Exception as e:
    print("Erreur de vol :", e)
    try: drone.land()
    except Exception: pass

## 10. Mission d'exploration autonome (réel)

`simulation_mode=False`. **On réutilise l'instance `drone` déjà connectée** (sections 7–9) via
`ExplorationMission(..., drone=drone)`. Aucun second socket UDP, aucun host codé en dur.

> **⚠️ Pré-requis terrain pour éviter `error No valid imu`.** Le Tello se stabilise grâce à sa
> caméra ventrale + IMU. Vole **au-dessus d'un sol mat et texturé** (tapis à motifs, journaux étalés…),
> **bien éclairé**, sans reflets. Sur sol uniforme/brillant ou en faible lumière, le drone refuse
> rotations et avances. Pense aussi à **désactiver la bascule Wi-Fi automatique** de Windows pour ne pas
> perdre le réseau du drone en vol (sinon `WinError 10051`).
>
> Garde-fous ajoutés : vérification IMU après décollage (`require_imu_ready`), abandon + atterrissage
> automatique après `max_consecutive_failures` échecs, pause de stabilisation `move_settle_time` entre
> mouvements, et `land()` ré-essayé plusieurs fois.

In [ ]:
# --- Garde de pré-requis (évite NameError/AttributeError cryptiques) ---
import tello_controller as _tc
_missing = [n for n in ("MissionConfig", "ExplorationMission", "MissionStatus", "TELLO_HOST", "drone")
            if n not in globals()]
if _missing:
    raise RuntimeError(
        f"Cellules manquantes dans ce kernel : {_missing}. "
        "Réexécute dans l'ordre : imports → config TELLO_HOST → section 7 (connexion)."
    )
if not hasattr(_tc.TelloController, "is_imu_ready"):
    raise RuntimeError(
        "tello_controller.py est une ANCIENNE version (is_imu_ready absent). "
        "Mets à jour TOUS les .py au même endroit, puis redémarre le kernel."
    )

# NE PAS faire drone.end() ici : on réutilise l'instance déjà connectée.
# (Si tu as lancé la boucle vidéo de la section 8, sors-en avec 'q' avant.)

config_real = MissionConfig(
    area_width=400, area_height=400,
    exploration_altitude=120,
    step_size=50, pattern="snake",
    scan_interval=200, safety_margin=80,
    min_battery=20,
    max_duration=180,
    # --- Robustesse vol réel ---
    require_imu_ready=True,        # refuse de poursuivre si l'IMU n'est pas prêt
    max_consecutive_failures=5,    # abandon + atterrissage après 5 échecs d'affilée
    move_settle_time=1.5,          # pause (s) entre mouvements pour reverrouiller l'IMU
    host=TELLO_HOST,   # cohérence ; ignoré ici car on fournit l'instance 'drone' ci-dessous
)

# Réutilisation de l'instance 'drone' connectée en section 7
mission_real = ExplorationMission(config_real, simulation_mode=False, drone=drone)

def on_status_r(old, new): print(f"[{time.strftime('%H:%M:%S')}] {old.value} → {new.value}")
def on_thermal_r(pos, temp, hs): print(f"  🔥 {temp:.0f}°C à ({pos.x:.0f}, {pos.y:.0f})")
waypoints_reached_real = []
def on_waypoint_r(wp, progress):
    waypoints_reached_real.append(wp)
mission_real.on_status_change = on_status_r
mission_real.on_thermal_alert = on_thermal_r
mission_real.on_waypoint_reached = on_waypoint_r

if mission_real.prepare_mission():
    print(f"✅ Prête — {mission_real.total_waypoints} waypoints")
    mission_real.start_exploration()
    try:
        start = time.time()
        while mission_real.status == MissionStatus.IN_PROGRESS and (time.time()-start) < config_real.max_duration:
            time.sleep(2)
    except KeyboardInterrupt:
        print("⚠️ Interruption — arrêt")
    finally:
        mission_real.stop_exploration()      # atterrissage + streamoff (instance NON fermée)

    rapport = mission_real.get_mission_report()
    print("\n--- Rapport ---")
    print("Statut          :", rapport["status"])
    if rapport.get("abort_reason"):
        reason = rapport["abort_reason"]
        if reason == "imu_not_ready":
            print("⚠️ Abandon: IMU non prêt → change de surface de vol (sol texturé) et l'éclairage.")
        elif reason == "too_many_failures":
            print("⚠️ Abandon: trop d'échecs (No valid imu / perte réseau).")
        else:
            print("⚠️ Abandon:", reason)
    print(f"Waypoints       : {rapport['waypoints']['completed']}/{rapport['waypoints']['total']}")
    print(f"Couverture      : {rapport['mapping']['coverage']:.1f}%")
    print(f"Température max  : {rapport['thermal']['max_temperature']:.1f}°C | feu: {rapport['thermal']['fire_detected']}")
    print(f"Batterie        : {rapport['drone']['battery']}%")
else:
    print("❌ Préparation impossible (batterie ? connexion ? IMU ? instance déjà fermée ?)")

### 10.1 Visualisation de la cartographie (réel)

Mêmes cartes que la Partie A (altitude / thermique / occupation / trajectoire), mais
construites à partir des données réellement collectées par `mission_real` pendant le vol.

> Si la mission a été abandonnée tôt (ex. `imu_not_ready`), les cartes seront quasi vides :
> c'est normal, peu de points ont été enregistrés. Refais voler au-dessus d'un sol texturé,
> batterie chargée, pour obtenir une vraie couverture.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

dm = mission_real.dual_map
ext = [-config_real.area_width/2, config_real.area_width/2,
       -config_real.area_height/2, config_real.area_height/2]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Altitude
ax = axes[0, 0]
im = ax.imshow(np.nan_to_num(dm.altitude_grid, nan=0), cmap='terrain', origin='lower', extent=ext)
plt.colorbar(im, ax=ax, label='Altitude (cm)'); ax.set_title("Carte d'altitude (réel)")
for o in dm.obstacles:
    ax.plot(o.x, o.y, 'X', color=('red' if o.is_mobile else 'black'), ms=10)

# 2. Thermique
ax = axes[0, 1]
im = ax.imshow(dm.thermal_grid, cmap='hot', origin='lower', extent=ext, vmin=20, vmax=200)
plt.colorbar(im, ax=ax, label='°C'); ax.set_title("Carte thermique (réel)")
for z in dm.thermal_zones:
    ax.add_patch(plt.Circle((z.x, z.y), z.radius, fill=False, color='cyan', lw=2))

# 3. Occupation
ax = axes[1, 0]
cmap = ListedColormap(['lightgray', 'lightgreen', 'red'])
norm = BoundaryNorm([-1.5, -0.5, 0.5, 100.5], cmap.N)
ax.imshow(dm.occupancy_grid, cmap=cmap, norm=norm, origin='lower', extent=ext)
ax.set_title("Grille d'occupation (réel)")
ax.legend(handles=[Patch(facecolor='lightgray', label='Inexploré'),
                   Patch(facecolor='lightgreen', label='Libre'),
                   Patch(facecolor='red', label='Occupé')], loc='upper right')

# 4. Trajectoire réellement parcourue
ax = axes[1, 1]
if waypoints_reached_real:
    tx = [w[0] for w in waypoints_reached_real]
    ty = [w[1] for w in waypoints_reached_real]
    ax.plot(tx, ty, 'b-', alpha=.6, lw=1)
    ax.scatter(tx, ty, c=np.linspace(0, 1, len(tx)), cmap='viridis', s=20)
    ax.plot(tx[0], ty[0], 'go', ms=12, label='Départ')
    ax.plot(tx[-1], ty[-1], 'r^', ms=12, label='Fin')
    ax.legend(loc='upper right')
else:
    ax.text(0.5, 0.5, "Aucun waypoint atteint\n(mission abandonnée tôt ?)",
            ha='center', va='center', transform=ax.transAxes)
for o in dm.obstacles:
    ax.add_patch(plt.Circle((o.x, o.y), o.radius,
                 color=('red' if o.is_mobile else 'orange'), alpha=.5))
ax.set_xlim(ext[0]-50, ext[1]+50); ax.set_ylim(ext[2]-50, ext[3]+50)
ax.set_aspect('equal'); ax.grid(alpha=.3); ax.set_title("Trajectoire (réel)")

for a in axes.flat:
    a.set_xlabel('X (cm)'); a.set_ylabel('Y (cm)')
plt.tight_layout(); plt.show()

# Cartes ASCII en complément
print(dm.to_ascii_map((mission_real.controller.position.x,
                       mission_real.controller.position.y), show_thermal=False))
print()
print(dm.to_ascii_map((mission_real.controller.position.x,
                       mission_real.controller.position.y), show_thermal=True))

### 10.2 Arrêt d'urgence (réel)
À garder sous la main : coupe immédiatement les moteurs.

In [ ]:
# mission_real.emergency_stop()   # ⚠️ chute immédiate du drone

### 10.3 Fermeture de la connexion (réel)

À lancer une fois la mission terminée pour libérer proprement le socket UDP.

In [ ]:
# Fermeture finale de l'instance drone (à exécuter quand tu as terminé)
try:
    drone.end()
    print("✅ Instance drone fermée")
except Exception as e:
    print("Déjà fermée ou erreur :", e)

---
**Récapitulatif des réglages terrain intégrés**

| Réglage | Valeur |
|---|---|
| Host de connexion | `None` → auto `192.168.10.1` (mode point d'accès) |
| Mode station | passer une IP explicite, ex. `192.168.0.101` |
| Instance drone | **réutilisée** entre sections (1 seul client UDP) |
| Délai post-`streamon` | `time.sleep(2)` |
| Lecture frames | `get_frame_read()` après le délai |
| Résolution caméra | 960×720 → shape `(720, 960, 3)` |
| Affichage | conversion **BGR→RGB** |
| Nettoyage | `streamoff()` ; `end()` seulement si instance interne |
| djitellopy | `>= 2.5.0` |
